# Grey Wolf

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

MODEL_PATH = '/kaggle/input/datasets/zeyadsheriif25/efficientnetb0-model/efficientnet_model.keras'
WORKING_DIR = '/kaggle/working'
EXTRACTED_DIR = os.path.join(WORKING_DIR, 'AI_Tourist_Dataset')
test_dir = os.path.join(EXTRACTED_DIR, 'test')

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

print("Loading test dataset...")
test_ds = image_dataset_from_directory(
    test_dir,
    label_mode='categorical',
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

print("Loading EfficientNet model...")
model = tf.keras.models.load_model(MODEL_PATH)

print("Generating predictions...")
predictions = model.predict(test_ds)
true_labels = np.concatenate([y for x, y in test_ds], axis=0)

y_true = np.argmax(true_labels, axis=1)
y_pred_probs = np.max(predictions, axis=1)
y_pred_classes = np.argmax(predictions, axis=1)

Loading test dataset...
Found 30 files belonging to 10 classes.
Loading EfficientNet model...
Generating predictions...
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step


In [ ]:
def fitness_function(threshold, y_true, y_pred_probs, y_pred_classes):
    accepted_mask = y_pred_probs >= threshold

    if np.sum(accepted_mask) == 0:
        return 0.0

    accepted_true = y_true[accepted_mask]
    accepted_pred = y_pred_classes[accepted_mask]

    correct = np.sum(accepted_true == accepted_pred)
    incorrect = np.sum(accepted_true != accepted_pred)

    score = correct - (incorrect * 2.0)
    return score

class GreyWolfOptimizer:
    def __init__(self, obj_func, lb, ub, search_agents, max_iter):
        self.obj_func = obj_func
        self.lb = lb
        self.ub = ub
        self.search_agents = search_agents
        self.max_iter = max_iter

        self.alpha_pos = 0.0
        self.alpha_score = float('-inf')

        self.beta_pos = 0.0
        self.beta_score = float('-inf')

        self.delta_pos = 0.0
        self.delta_score = float('-inf')

    def optimize(self, y_true, y_pred_probs, y_pred_classes):
        positions = np.random.uniform(0, 1, self.search_agents) * (self.ub - self.lb) + self.lb

        for l in range(self.max_iter):
            for i in range(self.search_agents):
                positions[i] = np.clip(positions[i], self.lb, self.ub)

                fitness = self.obj_func(positions[i], y_true, y_pred_probs, y_pred_classes)

                if fitness > self.alpha_score:
                    self.delta_score, self.delta_pos = self.beta_score, self.beta_pos
                    self.beta_score, self.beta_pos = self.alpha_score, self.alpha_pos
                    self.alpha_score, self.alpha_pos = fitness, positions[i]
                elif fitness > self.beta_score:
                    self.delta_score, self.delta_pos = self.beta_score, self.beta_pos
                    self.beta_score, self.beta_pos = fitness, positions[i]
                elif fitness > self.delta_score:
                    self.delta_score, self.delta_pos = fitness, positions[i]

            a = 2 - l * ((2) / self.max_iter)

            for i in range(self.search_agents):
                r1, r2 = np.random.rand(), np.random.rand()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * self.alpha_pos - positions[i])
                X1 = self.alpha_pos - A1 * D_alpha

                r1, r2 = np.random.rand(), np.random.rand()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * self.beta_pos - positions[i])
                X2 = self.beta_pos - A2 * D_beta

                r1, r2 = np.random.rand(), np.random.rand()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * self.delta_pos - positions[i])
                X3 = self.delta_pos - A3 * D_delta

                positions[i] = (X1 + X2 + X3) / 3

            if l % 10 == 0:
                print(f"Iteration {l}: Best Threshold (Alpha) = {self.alpha_pos:.4f}, Fitness Score = {self.alpha_score}")

        return self.alpha_pos

In [ ]:
SEARCH_AGENTS = 20
MAX_ITERATIONS = 150

print("Starting Grey Wolf Optimization...")
gwo = GreyWolfOptimizer(
    obj_func=fitness_function,
    lb=0.85,
    ub=1.0,
    search_agents=SEARCH_AGENTS,
    max_iter=MAX_ITERATIONS
)

optimal_threshold = gwo.optimize(y_true, y_pred_probs, y_pred_classes)

print("-" * 40)
print(f"GWO Optimized Confidence Threshold: {optimal_threshold:.4f}")
print("-" * 40)

accepted_mask = y_pred_probs >= optimal_threshold
accepted_count = np.sum(accepted_mask)
total_count = len(y_true)

print(f"Backend Behavior with Threshold {optimal_threshold:.4f}:")
print(f"Images Accepted: {accepted_count} / {total_count} ({(accepted_count/total_count)*100:.2f}%)")
print(f"Images Rejected: {total_count - accepted_count} / {total_count}")

Starting Grey Wolf Optimization...
Iteration 0: Best Threshold (Alpha) = 0.8830, Fitness Score = 26.0
Iteration 10: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 20: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 30: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 40: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 50: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 60: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 70: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 80: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 90: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 100: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 110: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 120: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 130: Best Threshold (Alpha) = 0.9978, Fitness Score = 27.0
Iteration 